## Split up one main table into many .csv files

In [1]:
import pandas as pd

In [2]:
# read
data = pd.read_csv('dataset.csv')

In [3]:
# Remove the Unnamed (the owner must have forgot to save without index)
data = data.drop(columns=['Unnamed: 0'])
data.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


In [4]:
genres = data['track_genre'].unique()

In [5]:
genres # No repeats or misspelled genres

array(['acoustic', 'afrobeat', 'alt-rock', 'alternative', 'ambient',
       'anime', 'black-metal', 'bluegrass', 'blues', 'brazil',
       'breakbeat', 'british', 'cantopop', 'chicago-house', 'children',
       'chill', 'classical', 'club', 'comedy', 'country', 'dance',
       'dancehall', 'death-metal', 'deep-house', 'detroit-techno',
       'disco', 'disney', 'drum-and-bass', 'dub', 'dubstep', 'edm',
       'electro', 'electronic', 'emo', 'folk', 'forro', 'french', 'funk',
       'garage', 'german', 'gospel', 'goth', 'grindcore', 'groove',
       'grunge', 'guitar', 'happy', 'hard-rock', 'hardcore', 'hardstyle',
       'heavy-metal', 'hip-hop', 'honky-tonk', 'house', 'idm', 'indian',
       'indie-pop', 'indie', 'industrial', 'iranian', 'j-dance', 'j-idol',
       'j-pop', 'j-rock', 'jazz', 'k-pop', 'kids', 'latin', 'latino',
       'malay', 'mandopop', 'metal', 'metalcore', 'minimal-techno', 'mpb',
       'new-age', 'opera', 'pagode', 'party', 'piano', 'pop-film', 'pop',
       'pow

## Add columns:
    genre_id, genre_name (already exists), description
    gemini was used to add decsriptions to the genres

In [7]:
import pathlib
from pathlib import Path
root_dir = Path('grouped_genres')

# collect csv files
csv_files = list(root_dir.glob('*.csv'))

dfs = []
for csv in csv_files:
    df = pd.read_csv(csv)
    # normalize column names
    df.columns = [c.strip() for c in df.columns]
    col_map = {}
    for c in df.columns:
        if 'genre' in c.lower():
            col_map[c] = 'genre'
        elif 'description' in c.lower():
            col_map[c] = 'short_description'
    df = df.rename(columns=col_map)
    parent = csv.stem
    df['parent_genre'] = parent
    # ensure required columns exist (fill missing with NA)
    for required in ['genre','short_description']:
        if required not in df.columns:
            df[required] = pd.NA
    dfs.append(df[['parent_genre','genre','short_description']])

all_genres = pd.concat(dfs, ignore_index=True) if dfs else pd.DataFrame(columns=['parent_genre','genre','short_description'])
all_genres['genre_id'] = range(1, len(all_genres)+1)
all_genres.to_csv('all_genres.csv', index=False)
all_genres.head()

,parent_genre,genre,short_description,genre_id
0,Hip-Hop,hip-hop,Music with stylized rhythmic and rhyming speec...,1
1,"Jazz, Blues & Funk",blues,"Genre rooted in African-American music, charac...",2
2,"Jazz, Blues & Funk",funk,"Rhythmic, danceable music emphasizing bass, dr...",3
3,"Jazz, Blues & Funk",honky-tonk,"Subgenre of country music focused on drinking,...",4
4,"Jazz, Blues & Funk",jazz,Improvised music with roots in ragtime and blu...,5
